# 20 - User-Defined Functions (UDFs) & Stored Procedures

**The Complete Interview & Practical Guide for Modern Data Engineers (Databricks, Spark & SQL)**

In enterprise data engineering, business logic must often be encapsulated for reuse, governance, and transactional integrity. 
This notebook covers every dimension tested in Senior & Mid-Level Data Engineering interviews:
- **The Core Architecture Matrix:** View vs. SQL Function vs. PySpark UDF vs. Vectorized Pandas UDF vs. Stored Procedure.
- **SQL User-Defined Functions (Scalar & Table UDTFs):** Reusable business metrics, Prompt Pay penalties, and calendar expansion.
- **Higher-Order Functions in Spark SQL:** Why `TRANSFORM`, `FILTER`, and `AGGREGATE` eliminate the dreaded `explode()` anti-pattern.
- **PySpark UDFs vs. Pandas Vectorized UDFs (PyArrow):** The JVM-to-Python serialization bottleneck and how Arrow vectorization achieves 10x-100x speedups.
- **Unity Catalog Security Functions:** Dynamic Data Masking (`MASK`) and Row-Level Security (`ROW FILTER`) with `is_account_group_member()`.
- **Stored Procedures & Procedural SQL:** Parameters (`IN`, `OUT`), variables (`DECLARE`), branching (`IF/ELSE`), dynamic SQL (`EXECUTE IMMEDIATE`), and ACID transactions (`BEGIN`, `COMMIT`, `ROLLBACK`).
- **The #1 Senior DE Migration Question:** Refactoring legacy row-by-row cursors and while-loops from Oracle/SQL Server into vectorized PySpark Window functions.


In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, assert_grain, tables, columns, explain, display,
    get_spark, get_dbutils, connect
)
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
dbutils = get_dbutils()
print("Setup complete. Local DuckDB claims warehouse and helper tools ready.")


---
### Section 1: The Big Picture — Architectural Comparison Matrix

In interview loops, architectural decision-making is evaluated with questions like:
> *"When do you write a Stored Procedure versus a UDF, a View, or an external PySpark pipeline?"*

| Construct | Purpose | Modifies Data (DML)? | Control Flow (`IF`/`LOOP`)? | Returns | Execution Engine | Best Use Case |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **View** | Query shortcut & schema abstraction | ❌ No (read-only) | ❌ No | Virtual table | SQL Engine | Standardizing joins, decoupling downstream queries from storage |
| **Scalar UDF** | Reusable row-level computation | ❌ No (pure function) | ⚠️ Limited | Single scalar value | SQL / Spark Catalyst | Business logic formulas (Prompt Pay interest, currency math, regex parsing) |
| **Table UDF (UDTF)** | Expand/transform input into rows | ❌ No | ⚠️ Limited | Relation (0 to N rows) | SQL / Spark | Unnesting complex types, date range explosion, calendar generation |
| **Stored Procedure** | Multi-step procedural orchestration | ✅ **Yes** (`INSERT`, `UPDATE`, `MERGE`, DDL) | ✅ **Yes** (`IF`, `WHILE`, `TRY...CATCH`) | Status code, OUT params, or Result Sets | Database Engine | In-warehouse batch ingestion, idempotent partition swaps, administrative tasks |
| **PySpark / dbt** | Distributed ETL / Medallion Lakehouse | ✅ Yes | ✅ Yes (Python / Jinja) | Delta Tables / Parquet | Spark Clusters | Heavy transformations, multi-terabyte ingestion, ML feature engineering |

#### Key Takeaway for Interviews:
* **Functions (UDFs)** are **pure calculations** that return values and **cannot have side effects** (cannot write or alter tables).
* **Stored Procedures** are designed to **execute workflows with side effects** (transactions, DML, DDL, table drops/swaps).


---
### Section 2: SQL User-Defined Functions (Scalar UDFs & UDTFs)

#### 2.1 Scalar SQL UDFs
A scalar function takes input parameters, runs deterministic or non-deterministic computation, and returns a single value.

**Databricks / ANSI SQL Syntax:**
```sql
CREATE OR REPLACE FUNCTION calculate_prompt_pay_interest(
    p_billed_amount DOUBLE,
    p_days_late INT
)
RETURNS DOUBLE
DETERMINISTIC
RETURN CASE 
    WHEN p_days_late <= 30 THEN 0.0
    ELSE ROUND(p_billed_amount * (0.12 / 365.0) * (p_days_late - 30), 2)
END;
```

Let's test this in DuckDB using DuckDB Macros / Functions on our live claims warehouse:


In [ ]:
# Create a scalar SQL function (Macro) for Prompt Pay statutory interest
# Rule: If adjudication exceeds 30 days, apply 12% statutory annual interest per day late.
con = connect()

con.execute('''
CREATE OR REPLACE TEMP MACRO fn_prompt_pay_interest(billed_amount, days_to_adjudicate) AS
    CASE 
        WHEN days_to_adjudicate <= 30 THEN 0.0
        ELSE ROUND(billed_amount * (0.12 / 365.0) * (days_to_adjudicate - 30), 2)
    END;
''')

# Query claims using our new function
df_penalties = q('''
SELECT 
    claim_id,
    provider_id,
    billed_amount,
    submitted_date,
    updated_at::DATE AS adjudication_date,
    (updated_at::DATE - submitted_date) AS days_elapsed,
    fn_prompt_pay_interest(billed_amount, updated_at::DATE - submitted_date) AS statutory_penalty
FROM claims
WHERE updated_at IS NOT NULL AND submitted_date IS NOT NULL
ORDER BY statutory_penalty DESC
LIMIT 10;
''')

display(df_penalties)


#### 2.2 Table-Valued User-Defined Functions (UDTFs)
While a scalar UDF returns a single value, a **Table Function (UDTF)** returns an entire set of rows.
UDTFs are ideal for:
1. Generating date/calendar dimensions on the fly.
2. Expanding financial schedules (e.g. amortizing a claim over 12 billing cycles).
3. Custom unnesting and pivoting without complex join syntax.

**Databricks SQL Syntax:**
```sql
CREATE OR REPLACE FUNCTION generate_date_range(start_date DATE, end_date DATE)
RETURNS TABLE (calendar_date DATE, day_of_week STRING)
RETURN SELECT 
    calendar_date,
    date_format(calendar_date, 'EEEE') AS day_of_week
FROM range(start_date, end_date + 1);
```

Let's test a Table Function in DuckDB that expands monthly claim aging buckets:


In [ ]:
# Table-Valued Function (UDTF) simulation: Generating Aging Buckets
con.execute('''
CREATE OR REPLACE TEMP MACRO udtf_aging_buckets() AS TABLE
    SELECT 1 AS bucket_id, '0-30 Days (Current)' AS bucket_name, 0 AS min_days, 30 AS max_days
    UNION ALL
    SELECT 2, '31-60 Days (Late)', 31, 60
    UNION ALL
    SELECT 3, '61-90 Days (Critical)', 61, 90
    UNION ALL
    SELECT 4, '90+ Days (Regulatory Breach)', 91, 9999;
''')

# Join claims against the table function to categorize unadjudicated claims as of '2025-07-01'
df_aging = q('''
WITH pending_claims AS (
    SELECT 
        claim_id,
        billed_amount,
        ('2025-07-01'::DATE - claim_date) AS age_days
    FROM claims
    WHERE status = 'Submitted'
)
SELECT 
    b.bucket_name,
    COUNT(c.claim_id) AS pending_claims,
    ROUND(SUM(c.billed_amount), 2) AS total_at_risk_dollars
FROM pending_claims c
JOIN udtf_aging_buckets() b 
    ON c.age_days BETWEEN b.min_days AND b.max_days
GROUP BY b.bucket_id, b.bucket_name
ORDER BY b.bucket_id;
''')

display(df_aging)


---
### Section 3: Spark SQL Higher-Order Functions (The "Zero-Explode" Revolution)

#### Why `explode()` is an Interview Red Flag:
In legacy Spark code, working with arrays (e.g. multiple diagnosis codes or service lines per claim) required `explode()`:
```python
# ❌ LEGACY ANTI-PATTERN: Exploding arrays
exploded = claims.select("claim_id", F.explode("diagnosis_codes").alias("diag"))
filtered = exploded.filter(F.col("diag").startswith("E11"))
result = filtered.groupBy("claim_id").agg(F.collect_list("diag")) # Expensive shuffle!
```
**Why this hurts at scale:**
1. **Row Explosion (Fan-Out):** A table with 10M rows and 10 codes per row explodes into **100M rows** in memory.
2. **Shuffle Overhead:** The subsequent `groupBy` forces a full cluster-wide hash partition shuffle across worker nodes.
3. **Memory Spikes:** High risk of executor `OutOfMemoryError` (OOM).

#### The Modern Spark Solution: Higher-Order Functions
Higher-Order Functions evaluate expressions **directly inside the array/map structure in-place** without changing the row grain!
- **`TRANSFORM(arr, x -> expr)`**: Applies transformation to each element (like Python `map`).
- **`FILTER(arr, x -> condition)`**: Filters elements matching predicate (like Python `filter`).
- **`EXISTS(arr, x -> condition)`**: Returns `TRUE` if any element satisfies condition.
- **`AGGREGATE(arr, zero, (acc, x) -> expr)`**: In-place reduction (like Python `reduce`).

Let's demonstrate Higher-Order Functions on claim diagnoses:


In [ ]:
# Higher-Order Functions Demonstration
# Create sample dataset with nested arrays of diagnosis codes
con.execute('''
CREATE OR REPLACE TEMP TABLE claim_diagnoses_nested AS
SELECT 
    'CLM001' AS claim_id, ['E11.9', 'I10', 'Z79.4'] AS icd10_codes, [120.0, 450.0, 80.0] AS line_amounts
UNION ALL
SELECT 
    'CLM002', ['I10', 'M54.5'], [85.0, 310.0]
UNION ALL
SELECT 
    'CLM003', ['E11.65', 'E11.9', 'N18.3'], [650.0, 210.0, 940.0];
''')

# 1. FILTER: Keep only Diabetes codes (ICD-10 starting with 'E11') without exploding
# 2. TRANSFORM: Standardize code format to upper-case prefixed
# 3. EXISTS: Check if Hypertension ('I10') is present
df_hof = q('''
SELECT 
    claim_id,
    icd10_codes,
    -- FILTER without explode
    list_filter(icd10_codes, x -> x LIKE 'E11%') AS diabetes_codes,
    -- TRANSFORM without explode
    list_transform(icd10_codes, x -> 'ICD10-' || x) AS prefixed_codes,
    -- EXISTS without explode
    list_contains(icd10_codes, 'I10') AS has_hypertension
FROM claim_diagnoses_nested;
''')

display(df_hof)


---
### Section 4: PySpark UDF vs. Vectorized Pandas UDF (PyArrow)

This is one of the most frequently asked Spark performance questions in technical interviews:
> *"Why are standard PySpark UDFs slow, and how do Vectorized Pandas UDFs solve this?"*

```
STANDARD PYTHON @udf (Row-by-Row):
[JVM Executor] --(Py4J / Socket Serialization)--> [Python Worker] 
(1 row at a time, un-optimized bytecode, Tungsten bypassed, GC pressure)

VECTORIZED @pandas_udf (Batch / Apache Arrow):
[JVM Executor] ======(Zero-Copy PyArrow Buffers)======> [Python Worker (Pandas / SIMD)]
(Chunks of thousands of rows vectorized simultaneously in C memory)
```

#### The Performance Hierarchy:
1. 🥇 **Native Spark SQL Functions (`pyspark.sql.functions`):** 100% JVM execution, Catalyst optimizer can fold constants and push predicates down, Tungsten off-heap memory. Always use native functions first!
2. 🥈 **Vectorized Pandas UDF (`@pandas_udf`):** Runs via PyArrow batch serialization. 10x-100x faster than standard Python UDFs.
3. 🥉 **Standard PySpark UDF (`@udf`):** Row-by-row Python worker IPC serialization. Massive overhead. Avoid whenever possible!

Let's look at the exact code syntax interviewers expect:


In [ ]:
# Code Demonstration: Standard UDF vs Vectorized Pandas UDF Syntax

# 1. Standard Python UDF (Row-by-Row IPC)
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType, DoubleType

@udf(returnType=StringType())
def categorize_risk_standard(billed: float, paid: float) -> str:
    if billed is None or paid is None:
        return "UNKNOWN"
    ratio = paid / billed if billed > 0 else 0.0
    if ratio < 0.20:
        return "HIGH_AUDIT_RISK"
    elif ratio < 0.60:
        return "MEDIUM_RISK"
    return "LOW_RISK"

# 2. Vectorized Pandas UDF (PyArrow Batched Vectorization)
from pyspark.sql.functions import pandas_udf
import pandas as pd

@pandas_udf(StringType())
def categorize_risk_vectorized(billed_series: pd.Series, paid_series: pd.Series) -> pd.Series:
    ratio = paid_series / billed_series.replace(0, float('nan'))
    results = pd.Series("LOW_RISK", index=billed_series.index)
    results[ratio < 0.60] = "MEDIUM_RISK"
    results[ratio < 0.20] = "HIGH_AUDIT_RISK"
    results[billed_series.isna() | paid_series.isna()] = "UNKNOWN"
    return results

print("Vectorized Pandas UDF and standard UDF syntax defined.")
print("Interview tip: Always specify @pandas_udf with explicit return type and pandas Series signature!")


---
### Section 5: Unity Catalog Functions (Column Masks & Row Filters)

In Databricks Unity Catalog, functions are first-class citizens registered in the three-level namespace: `catalog.schema.function_name`.
They form the backbone of **Attribute-Based Access Control (ABAC)** and **HIPAA/PHI Data Masking**.

#### 5.1 Dynamic Column Masking
Instead of duplicating tables for different user groups, a single table is secured with a masking function.

**Unity Catalog Syntax:**
```sql
-- 1. Create the masking function
CREATE OR REPLACE FUNCTION mask_patient_ssn(ssn STRING)
RETURNS STRING
RETURN CASE 
    WHEN is_account_group_member('phi_compliance_auditors') THEN ssn
    ELSE 'XXX-XX-' || RIGHT(ssn, 4)
END;

-- 2. Apply the mask to the target table column
ALTER TABLE silver.members ALTER COLUMN ssn SET MASK mask_patient_ssn;
```

#### 5.2 Dynamic Row-Level Security (Row Filters)
Row filter functions return a boolean `TRUE/FALSE` determining whether the querying user can see the row.

**Unity Catalog Syntax:**
```sql
-- 1. Create the row filter function
CREATE OR REPLACE FUNCTION filter_claims_by_region(region STRING)
RETURNS BOOLEAN
RETURN 
    is_account_group_member('enterprise_admin') OR 
    region = current_user_region();

-- 2. Bind filter to table
ALTER TABLE silver.claims SET ROW FILTER filter_claims_by_region ON (region);
```

Let's test dynamic masking simulation on our members table in DuckDB:


In [ ]:
# Masking Simulation in DuckDB
con.execute('''
CREATE OR REPLACE TEMP MACRO mask_member_phi(member_name, is_compliance_user) AS
    CASE 
        WHEN is_compliance_user THEN member_name
        ELSE SUBSTRING(member_name, 1, 1) || '*** ' || SUBSTRING(member_name, POSITION(' ' IN member_name) + 1, 1) || '***'
    END;
''')

# Query as general user (is_compliance_user = FALSE)
df_masked = q('''
SELECT 
    member_id,
    first_name || ' ' || last_name AS raw_name,
    mask_member_phi(first_name || ' ' || last_name, FALSE) AS masked_name_standard_user,
    mask_member_phi(first_name || ' ' || last_name, TRUE) AS unmasked_name_compliance_user
FROM members
LIMIT 5;
''')

display(df_masked)


---
### Section 6: Stored Procedures & Procedural Logic

#### 6.1 Anatomy of a Stored Procedure
A stored procedure encapsulates multi-step workflows. Unlike UDFs, stored procedures can:
1. Accept `IN`, `OUT`, and `INOUT` parameters.
2. Declare local variables (`DECLARE`).
3. Execute conditional logic (`IF / ELSEIF / ELSE`).
4. Perform DML (`INSERT`, `UPDATE`, `DELETE`, `MERGE`) and DDL.
5. Control transactions (`BEGIN`, `COMMIT`, `ROLLBACK`).

**Databricks SQL / ANSI Scripting Syntax:**
```sql
CREATE OR REPLACE PROCEDURE sp_classify_provider(
    IN p_provider_id STRING,
    OUT p_total_claims INT,
    OUT p_total_paid DOUBLE,
    OUT p_tier STRING
)
LANGUAGE SQL
AS $$
DECLARE
    v_paid DOUBLE;
    v_claims INT;
BEGIN
    SELECT COUNT(*), COALESCE(SUM(paid_amount), 0.0)
    INTO v_claims, v_paid
    FROM claims
    WHERE provider_id = p_provider_id;

    IF v_paid >= 1000000 THEN
        p_tier := 'Tier 1 - High Volume';
    ELSEIF v_paid >= 250000 THEN
        p_tier := 'Tier 2 - Medium Volume';
    ELSE
        p_tier := 'Tier 3 - Standard';
    END IF;

    p_total_claims := v_claims;
    p_total_paid := v_paid;
END;
$$;
```

Let's test this procedural logic interactively:


In [ ]:
# Interactive Procedural Simulation: Classification with Parameters & Branching

def sp_classify_provider(provider_id: str) -> dict:
    """Simulates execution of stored procedure sp_classify_provider."""
    con = connect()
    res = con.execute('''
        SELECT COUNT(*), COALESCE(SUM(paid_amount), 0.0)
        FROM claims
        WHERE provider_id = ?
    ''', [provider_id]).fetchone()
    
    claim_count, total_paid = res[0], float(res[1])
    
    # Procedural branching
    if total_paid >= 1_000_000:
        tier = 'Tier 1 - High Volume'
    elif total_paid >= 250_000:
        tier = 'Tier 2 - Medium Volume'
    else:
        tier = 'Tier 3 - Standard'
        
    return {
        "provider_id": provider_id,
        "total_claims": claim_count,
        "total_paid": round(total_paid, 2),
        "status_tier": tier
    }

for pid in ["PRV0151", "PRV0020", "PRV0001"]:
    print(sp_classify_provider(pid))


#### 6.2 Idempotent Batch Reloads: The #1 Production Pattern
The core responsibility of stored procedures in data engineering pipelines is **idempotent partition processing**.
- ❌ **Non-Idempotent:** Appending records via `INSERT INTO target ...` (duplicates data on retry).
- ✅ **Idempotent Pattern 1 (Atomic Delete + Insert):**
  ```sql
  DELETE FROM gold_monthly_kpi WHERE partition_month = p_month;
  INSERT INTO gold_monthly_kpi SELECT ... WHERE partition_month = p_month;
  ```
- ✅ **Idempotent Pattern 2 (`MERGE INTO` Upsert):**
  Matches on primary key + partition key, updating existing rows and inserting new ones.

#### 6.3 Dynamic SQL (`EXECUTE IMMEDIATE`) & Warehouse Maintenance
Dynamic SQL builds query strings at runtime.
```sql
DECLARE v_table_name STRING DEFAULT 'claims_silver';
BEGIN
    -- Dynamically trigger optimization
    EXECUTE IMMEDIATE 'OPTIMIZE ' || v_table_name || ' ZORDER BY (claim_date, provider_id);';
END;
```
⚠️ **Security Note for Interviews:** Always sanitize input parameters or check against an allowed table whitelist to prevent **SQL Injection**!


In [ ]:
# Dynamic SQL Simulation: Metadata-Driven Table Partition Optimizer

def sp_dynamic_optimize_table(table_name: str, allowed_tables: list = None) -> str:
    """Demonstrates sanitized dynamic SQL execution."""
    if allowed_tables is None:
        allowed_tables = ["claims", "providers", "claim_services", "members"]
        
    # SQL Injection Guard
    if table_name not in allowed_tables:
        raise ValueError(f"Security Alert: Table '{table_name}' is not in allowed whitelist!")
        
    # Dynamically construct maintenance statement
    con = connect()
    dynamic_sql = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    row_count = con.execute(dynamic_sql).fetchone()[0]
    
    return f"Successfully executed dynamic maintenance on '{table_name}'. Verified {row_count:,} rows."

print(sp_dynamic_optimize_table("claims"))


---
### Section 7: The #1 Senior DE Interview Drill — Refactoring Legacy Cursor Loops to Vectorized PySpark

A classic senior interview question:
> *"We have a 500-line Oracle or SQL Server stored procedure that uses a CURSOR to loop row-by-row and calculate running totals or state transitions. How do you refactor this to PySpark/Databricks?"*

#### The Legacy RDBMS Anti-Pattern:
```sql
-- ❌ LEGACY RDBMS ANTI-PATTERN (Sequential Row-by-Row Cursor)
DECLARE @claim_id VARCHAR(20), @paid_amount DECIMAL(18,2), @running_total DECIMAL(18,2) = 0.0;
DECLARE claim_cursor CURSOR FOR 
    SELECT claim_id, paid_amount FROM claims WHERE provider_id = 'PRV0151' ORDER BY claim_date;

OPEN claim_cursor;
FETCH NEXT FROM claim_cursor INTO @claim_id, @paid_amount;
WHILE @@FETCH_STATUS = 0
BEGIN
    SET @running_total = @running_total + @paid_amount;
    UPDATE claims_summary SET cumulative_paid = @running_total WHERE claim_id = @claim_id;
    FETCH NEXT FROM claim_cursor INTO @claim_id, @paid_amount;
END;
CLOSE claim_cursor;
DEALLOCATE claim_cursor;
```

#### Why Cursors Kill Performance in Modern Data Warehouses & Spark:
1. **Single-Threaded Bottleneck:** Cursors execute on a single CPU core, ignoring distributed cluster nodes.
2. **Locking & I/O Overhead:** Row-by-row `UPDATE`s trigger hundreds of thousands of micro-transactions, bloating logs.
3. **No Vectorization:** Bypasses columnar SIMD hardware acceleration.

#### The Modern Solution: Vectorized Window Functions!
A single Window function in PySpark or modern SQL evaluates running totals across millions of rows in seconds:


In [ ]:
# Modern Vectorized Replacement for Legacy Cursors
# Calculates running cumulative paid amounts partitioned by provider_id ordered by date

df_vectorized = q('''
SELECT 
    provider_id,
    claim_id,
    claim_date,
    paid_amount,
    -- Replaces 30 lines of cursor loop logic with one vectorized calculation:
    ROUND(SUM(paid_amount) OVER (
        PARTITION BY provider_id 
        ORDER BY claim_date, claim_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ), 2) AS running_total_paid
FROM claims
WHERE provider_id = 'PRV0151'
ORDER BY claim_date, claim_id
LIMIT 10;
''')

display(df_vectorized)
assert_grain(df_vectorized, "claim_id")


---
### Section 8: Interactive Practice Drills ("YOUR TURN")

#### Drill 1: Create a Grain-Safe Scalar Function (Macro)
Write a function `fn_payment_accuracy(billed, paid)`:
- Returns `100.0 * (paid / billed)` rounded to 2 decimal places.
- Handles division by zero safely (if billed <= 0 or null, returns 0.0).
- Applies it across claims.


In [ ]:
# YOUR TURN - DRILL 1
# 1. Create the macro fn_payment_accuracy
con.execute('''
CREATE OR REPLACE TEMP MACRO fn_payment_accuracy(billed, paid) AS
    CASE 
        WHEN billed IS NULL OR billed <= 0 THEN 0.0
        WHEN paid IS NULL THEN 0.0
        ELSE ROUND((paid * 100.0) / billed, 2)
    END;
''')

# 2. Query top 5 claims with highest payment accuracy
drill1_result = q('''
SELECT 
    claim_id,
    billed_amount,
    paid_amount,
    fn_payment_accuracy(billed_amount, paid_amount) AS accuracy_pct
FROM claims
WHERE billed_amount > 1000
ORDER BY accuracy_pct DESC
LIMIT 5;
''')

display(drill1_result)
assert_grain(drill1_result, "claim_id")


#### Drill 2: Higher-Order Function Array Processing
Given claims with an array of charge amounts `[150.0, 45.0, 300.0, 12.0]`, use higher-order functions to:
1. Filter out minor charges (< $50.00).
2. Add a 10% administrative surcharge to all remaining charges.
3. Calculate the total billable amount.


In [ ]:
# YOUR TURN - DRILL 2: Higher-Order Array Transformation
con.execute('''
CREATE OR REPLACE TEMP TABLE claim_charges_test AS
SELECT 'CLM101' AS claim_id, [150.0, 45.0, 300.0, 12.0, 800.0] AS service_charges
UNION ALL
SELECT 'CLM102', [30.0, 40.0, 25.0]
UNION ALL
SELECT 'CLM103', [1200.0, 550.0, 80.0];
''')

drill2_result = q('''
SELECT 
    claim_id,
    service_charges,
    -- Step 1: Filter charges >= 50.0
    list_filter(service_charges, x -> x >= 50.0) AS major_charges,
    -- Step 2: Transform with 10% surcharge
    list_transform(
        list_filter(service_charges, x -> x >= 50.0), 
        x -> ROUND(x * 1.10, 2)
    ) AS surcharged_charges,
    -- Step 3: Sum the transformed charges (coalesce empty lists to 0.0)
    COALESCE(ROUND(list_sum(list_transform(
        list_filter(service_charges, x -> x >= 50.0), 
        x -> x * 1.10
    )), 2), 0.0) AS total_surcharged_sum
FROM claim_charges_test;
''')

display(drill2_result)
assert_grain(drill2_result, "claim_id")


#### Drill 3: Stored Procedure for Idempotent Monthly Ledger Refresh
Write a simulated stored procedure `sp_refresh_monthly_ledger(p_month)` that:
1. Clears target partition records for `p_month`.
2. Computes total claims, approved amount, and average payment per claim.
3. Inserts fresh records and returns execution audit metrics.


In [ ]:
# YOUR TURN - DRILL 3: Idempotent Stored Procedure Batch Refresh
con.execute('''
CREATE TEMP TABLE IF NOT EXISTS gold_monthly_ledger_target (
    month_start DATE PRIMARY KEY,
    total_claims BIGINT,
    total_billed DOUBLE,
    total_paid DOUBLE,
    avg_paid_per_claim DOUBLE,
    refreshed_at TIMESTAMP
);
''')

def sp_refresh_monthly_ledger(month_str: str) -> dict:
    con = connect()
    
    # 1. Atomic DELETE existing partition
    deleted = con.execute('''
        DELETE FROM gold_monthly_ledger_target WHERE month_start = ?::DATE
    ''', [month_str]).fetchone()[0]
    
    # 2. INSERT fresh aggregate
    con.execute('''
        INSERT INTO gold_monthly_ledger_target
        SELECT 
            date_trunc('month', claim_date)::DATE AS month_start,
            COUNT(*) AS total_claims,
            ROUND(SUM(billed_amount), 2) AS total_billed,
            ROUND(SUM(paid_amount), 2) AS total_paid,
            ROUND(AVG(paid_amount), 2) AS avg_paid_per_claim,
            CURRENT_TIMESTAMP AS refreshed_at
        FROM claims
        WHERE date_trunc('month', claim_date)::DATE = ?::DATE
        GROUP BY 1;
    ''', [month_str])
    
    row = con.execute('''
        SELECT total_claims, total_paid, avg_paid_per_claim 
        FROM gold_monthly_ledger_target 
        WHERE month_start = ?::DATE
    ''', [month_str]).fetchone()
    
    return {
        "partition_month": month_str,
        "rows_cleared_prior": deleted,
        "claims_computed": row[0] if row else 0,
        "total_paid_dollars": row[1] if row else 0.0,
        "avg_paid": row[2] if row else 0.0,
        "status": "SUCCESS"
    }

# Run first load
res_load1 = sp_refresh_monthly_ledger('2024-03-01')
print("Run 1 (Initial Load):", res_load1)

# Run immediate re-run (Idempotency test)
res_load2 = sp_refresh_monthly_ledger('2024-03-01')
print("Run 2 (Idempotent Retry):", res_load2)

# Verify table grain: exactly 1 row!
display(q("SELECT * FROM gold_monthly_ledger_target"))
assert_grain(q("SELECT * FROM gold_monthly_ledger_target"), "month_start")


---
### Section 9: Top 10 Stored Procedures & Functions Interview Question Bank

#### 1. What is the fundamental difference between a Stored Procedure and a User-Defined Function (UDF)?
* **Answer:** A UDF is a **pure function** that accepts inputs, executes a computation, and returns a scalar value or table. A UDF **cannot perform DML** (cannot insert, update, delete, or alter table schemas). A Stored Procedure is designed for **procedural workflows with side effects** (executing DDL/DML, orchestrating multi-table pipelines, controlling transactions with `BEGIN/COMMIT/ROLLBACK`, and returning status codes or multiple result sets).

#### 2. Why are native Spark SQL functions preferred over Python UDFs?
* **Answer:** Native Spark functions execute directly in the JVM with Tungsten off-heap memory management and whole-stage code generation. The Catalyst optimizer can inspect expressions, reorder operations, push predicates down to Parquet/Delta storage, and perform constant folding. Standard Python UDFs force row-by-row serialization across a Py4J socket to a separate Python process, creating massive IPC overhead and rendering the optimizer blind to the code.

#### 3. How does a Vectorized Pandas UDF (`@pandas_udf`) achieve 10x-100x speedup over a standard Python `@udf`?
* **Answer:** Instead of row-by-row serialization, Pandas UDFs use **Apache Arrow** for batched, columnar in-memory data transfer between the JVM and Python worker. Data is processed in large vectorized chunks using C-accelerated Pandas/NumPy SIMD operations with zero-copy deserialization.

#### 4. When would you use Spark SQL Higher-Order Functions instead of `explode()`?
* **Answer:** Whenever you need to transform, filter, or check conditions across array or map elements without changing the grain of the DataFrame. `explode()` duplicates parent rows, causing severe memory spikes, row fan-out, and requiring a subsequent shuffle-heavy `groupBy`. Higher-Order Functions (`TRANSFORM`, `FILTER`, `EXISTS`, `AGGREGATE`) execute inline in-place with **zero fan-out and zero shuffle**.

#### 5. How do you guarantee idempotency in a stored procedure batch pipeline?
* **Answer:** Either through **Atomic Partition Overwrites** (`DELETE FROM target WHERE partition_col = p_val` followed by `INSERT INTO target`) or through an atomic **`MERGE INTO`** statement matching on the primary key and partition column. Re-running the procedure for the same partition must result in the exact same state without duplicate rows.

#### 6. How do transactions work inside a stored procedure, and why are they critical?
* **Answer:** Using `BEGIN TRANSACTION`, `COMMIT`, and `ROLLBACK` within an exception block (`TRY...CATCH`). In multi-table pipelines (e.g. updating a financial ledger and inserting an audit log), if step 2 fails, `ROLLBACK` undoes all preceding operations so the warehouse is never left in a corrupt, partially written state.

#### 7. What is Dynamic SQL (`EXECUTE IMMEDIATE`) and what security precaution is mandatory?
* **Answer:** Dynamic SQL constructs query statements as strings at runtime and executes them dynamically (e.g. metadata-driven partition optimization `OPTIMIZE <table> ZORDER BY ...`). The mandatory precaution is preventing **SQL Injection** by validating table and column inputs against a strict whitelist or using parameterized queries.

#### 8. How do you refactor legacy cursors and while-loops from Oracle or SQL Server into Databricks/Spark?
* **Answer:** By replacing sequential, single-threaded row-by-row cursor logic with **vectorized Window functions** (`Window.partitionBy().orderBy().rowsBetween()`), joins, or conditional aggregations. This converts an $O(N)$ sequential process into a distributed, set-based parallel execution across all cluster worker nodes.

#### 9. What are Unity Catalog Column Masks and Row Filters, and how do they leverage SQL functions?
* **Answer:** They implement Attribute-Based Access Control (ABAC) at the storage layer. A user-defined SQL function containing conditional logic (e.g., `CASE WHEN is_account_group_member('pii_admin') THEN ssn ELSE 'XXX-XX-' || RIGHT(ssn, 4) END`) is bound to a table column (`SET MASK`) or table rows (`SET ROW FILTER`), dynamically redacting sensitive PHI/PII data without duplicating tables.

#### 10. When would you choose a Databricks Workflow / Python script over a Stored Procedure?
* **Answer:** Use Python / Databricks Workflows when tasks interact with external systems (calling REST APIs, downloading cloud blob storage files, sending Slack alerts), require third-party libraries (Scikit-Learn, Pandas, MLflow), or orchestrate complex DAG dependencies across multiple clusters. Use Stored Procedures when the data already resides entirely within the warehouse and operations benefit from SQL pushdown execution.
